In [34]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyampute.exploration.mcar_statistical_tests import MCARTest
import psycopg2
import getpass

from dotenv import load_dotenv
from configparser import ConfigParser
from sqlalchemy import create_engine

import os

In [2]:
# We seek to use a config file to avoid absolute file paths
file = 'config.ini' #config file of interest

config = ConfigParser()
config.read(file)

print(config.sections()) # outlines the sections in the file

# accesses the motor data element, in this case it's the file path for it
paths = [config['paths']['motor_data'], config['paths']['claims_data']]

#path
df_motor = pd.read_csv(paths[0], sep=";")
df_claims = pd.read_csv(paths[1], sep=";")

df_motor.columns.tolist(), df_claims.columns.tolist() # It's important to understand the columns that exist in the tables

['paths', 'postgres']


(['ID',
  'Date_start_contract',
  'Date_last_renewal',
  'Date_next_renewal',
  'Date_birth',
  'Date_driving_licence',
  'Distribution_channel',
  'Seniority',
  'Policies_in_force',
  'Max_policies',
  'Max_products',
  'Lapse',
  'Date_lapse',
  'Payment',
  'Premium',
  'Cost_claims_year',
  'N_claims_year',
  'N_claims_history',
  'R_Claims_history',
  'Type_risk',
  'Area',
  'Second_driver',
  'Year_matriculation',
  'Power',
  'Cylinder_capacity',
  'Value_vehicle',
  'N_doors',
  'Type_fuel',
  'Length',
  'Weight'],
 ['ID', 'Cost_claims_year', 'Cost_claims_by_type', 'Claims_type'])

In [3]:
temp_arr = [i for i in df_motor.columns if 'date' in i.lower() or 'year' in i.lower()]
temp_arr

['Date_start_contract',
 'Date_last_renewal',
 'Date_next_renewal',
 'Date_birth',
 'Date_driving_licence',
 'Date_lapse',
 'Cost_claims_year',
 'N_claims_year',
 'Year_matriculation']

In [6]:
# Converts all the dates and years into a format that can be calculated with during runtime
for col in temp_arr:
    if 'date' in col.lower():
        #print(col)
        df_motor[col] =  pd.to_datetime(df_motor[col], dayfirst=True)
    elif 'year' in col.lower():
        #print(col)
        df_motor[col] =  pd.to_numeric(df_motor[col])
        
# Converts the data types from the previous D and P to 1 and 0s so that it's in a usable format.
df_motor["Type_fuel"] = df_motor["Type_fuel"].map({"D": 0,"P": 1}) # we need to drop this from the dataset as we must binarise the dataset, con

test_frame = df_motor.copy()
#df_motor.info()

In [7]:
test_frame

,ID,Date_start_contract,Date_last_renewal,Date_next_renewal,Date_birth,Date_driving_licence,Distribution_channel,Seniority,Policies_in_force,Max_policies,...,Area,Second_driver,Year_matriculation,Power,Cylinder_capacity,Value_vehicle,N_doors,Type_fuel,Length,Weight
0,1,2015-11-05,2015-11-05,2016-11-05,1956-04-15,1976-03-20,0,4,1,2,...,0,0,2004,80,599,7068.00,0,NaN,NaN,190
1,1,2015-11-05,2016-11-05,2017-11-05,1956-04-15,1976-03-20,0,4,1,2,...,0,0,2004,80,599,7068.00,0,NaN,NaN,190
2,1,2015-11-05,2017-11-05,2018-11-05,1956-04-15,1976-03-20,0,4,2,2,...,0,0,2004,80,599,7068.00,0,NaN,NaN,190
3,1,2015-11-05,2018-11-05,2019-11-05,1956-04-15,1976-03-20,0,4,2,2,...,0,0,2004,80,599,7068.00,0,NaN,NaN,190
4,2,2017-09-26,2017-09-26,2018-09-26,1956-04-15,1976-03-20,0,4,2,2,...,0,0,2004,80,599,7068.00,0,NaN,NaN,190
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105550,53498,2018-07-30,2018-07-30,2019-07-30,1981-07-25,2007-02-14,0,1,1,1,...,0,0,2000,110,1997,24320.00,5,NaN,4.740,1480
105551,53499,2018-08-16,2018-08-16,2019-08-16,1976-12-08,2017-11-29,0,1,1,1,...,0,0,2013,129,1998,30861.97,5,NaN,4.650,1440
105552,53500,2018-11-21,2018-11-21,2019-11-21,1974-04-01,2011-10-05,0,1,1,1,...,0,0,1999,55,999,7800.00,5,NaN,3.495,830
105553,53501,2018-11-21,2018-11-21,2019-11-21,1946-09-15,1982-02-02,0,1,1,1,...,0,0,2004,90,1753,16610.00,5,NaN,4.555,1399


In [17]:
missing = test_frame.isna().astype(int)
missing_cov = missing.cov()
missing_cov

,ID,Date_start_contract,Date_last_renewal,Date_next_renewal,Date_birth,Date_driving_licence,Distribution_channel,Seniority,Policies_in_force,Max_policies,...,Area,Second_driver,Year_matriculation,Power,Cylinder_capacity,Value_vehicle,N_doors,Type_fuel,Length,Weight
ID,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0
Date_start_contract,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0
Date_last_renewal,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0
Date_next_renewal,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0
Date_birth,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0
Date_driving_licence,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0
Distribution_channel,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0
Seniority,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0
Policies_in_force,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0
Max_policies,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0


In [10]:
# Here we choose to make this a testing data frame as we don't want to lose these columns
#test_frame = df_motor.copy()
#test_frame.drop(columns=['Type_fuel','ID','Date_start_contract','Date_last_renewal', 'Date_next_renewal'], inplace=True)
#test_frame.dropna(inplace=True)

# We should replace all the date objects with age objects, with which we will be meaningful for the covariance calculation
#days_const = 365.25
#Dropped = ['Date_start_contract','Date_last_renewal', 'Date_next_renewal']

#for i in range(0,len(temp_arr)):
#    col = temp_arr[i]
#    if col in Dropped:
#        pass
#    elif 'date' in col.lower():
#        test_frame[f'Age{i}'] = (pd.Timestamp.today() - test_frame[col]).dt.days / days_const
#        test_frame.drop(columns=[col], inplace=True)

#test_frame

In [18]:
missing_cov.mean() # this is our mean vector

ID                      0.000000
Date_start_contract     0.000000
Date_last_renewal       0.000000
Date_next_renewal       0.000000
Date_birth              0.000000
Date_driving_licence    0.000000
Distribution_channel    0.000000
Seniority               0.000000
Policies_in_force       0.000000
Max_policies            0.000000
Max_products            0.000000
Lapse                   0.000000
Date_lapse              0.007438
Payment                 0.000000
Premium                 0.000000
Cost_claims_year        0.000000
N_claims_year           0.000000
N_claims_history        0.000000
R_Claims_history        0.000000
Type_risk               0.000000
Area                    0.000000
Second_driver           0.000000
Year_matriculation      0.000000
Power                   0.000000
Cylinder_capacity       0.000000
Value_vehicle           0.000000
N_doors                 0.000000
Type_fuel               0.000000
Length                  0.002977
Weight                  0.000000
dtype: flo

In [37]:
mt = MCARTest(method="little")

p_value = mt.little_mcar_test(test_frame)

print(p_value)

p_value = mt.little_mcar_test(test_frame)

TypeError: unsupported operand type(s) for +: 'numpy.float64' and 'Timedelta'